# CMPE492 — MimicTalk Batch (Stage 2) — CROSS-AUDIO VERSION

**Purpose:** Same as Stage 2, but with **cross-identity audio mapping** to enable a proper LP-only baseline.

**Cross-audio mapping (cyclic):**
- id01 video personalization + **id02 audio** for inference
- id02 video personalization + id03 audio
- ...
- id10 video personalization + id01 audio

**Why:** Previous experiment used each identity's OWN audio, so LP-only's raw video already had matching lip-sync. By using DIFFERENT audio, we ensure LP-only's lip motion does NOT match the audio — creating a proper control where MimicTalk's contribution can be measured.

**Input:** `hdtf_prepared.zip` from Stage 1
**Output:** `mimic_videos.zip` — each video driven by the NEXT identity's audio

---
**Run order:** 1 → 2 → 3 → 4 → 5 → 6 (same as before, but use the cross-audio cell 5)

## Cell 1 — Clone + Install (restart runtime after this)

In [ ]:
# ── CELL 1: Clone repo ──
!git clone https://github.com/yerfor/MimicTalk.git
%cd MimicTalk

# ── CELL 2: Install all dependencies (slow ~10 min) ──
# After this cell finishes → Runtime > Restart runtime → continue from Cell 3

import subprocess, sys

print("── [0] System libs for PyAudio ──")
!apt-get update -qq && apt-get install -y -qq libportaudio2 portaudio19-dev

print("── [1] PyTorch 2.4 (CUDA 12.1) ──")
!pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 \
    --index-url https://download.pytorch.org/whl/cu121

print("── [2] Patch requirements.txt ──")
!sed -i 's/numba==0.56.4/numba/'                    docs/prepare_env/requirements.txt
!sed -i 's/face-alignment==1.4.1/face-alignment/'   docs/prepare_env/requirements.txt
!sed -i 's/torchmetrics==0.11.4/torchmetrics/'       docs/prepare_env/requirements.txt
!sed -i 's/scipy==1.11.1/scipy/'                     docs/prepare_env/requirements.txt
!sed -i '/httpx==0.23.3/d'                           docs/prepare_env/requirements.txt

print("── [3] pip requirements ──")
!pip install -q -r docs/prepare_env/requirements.txt

print("── [4] openmim + mmcv ──")
!pip install -q cython openmim==0.3.9
!pip install -q --upgrade setuptools requests packaging rich filelock tqdm jedi
!mim install mmcv==2.1.0

print("── [5] PyTorch3D ──")
!pip install -q "git+https://github.com/facebookresearch/pytorch3d.git@stable"

print("── [6] Extra fixes ──")
!pip install -q --upgrade librosa
!pip install transformers==4.40.0 -q
!pip install -q gdown

print("\n✅ Done. Now do: Runtime > Restart runtime, then run Cell 3 onwards.")

## Cell 2 — Restore Env + Download Weights (after restart)

In [ ]:
# ── Restore working directory & env vars (run after every restart) ──
%cd MimicTalk
%env PYTHONPATH=.
%env HF_ENDPOINT=https://hf-mirror.com
print("CWD:", __import__('os').getcwd())

# ── BFM face model ──
!gdown --folder '1o4t5YIw7w4cMUN4bgU9nPf6IyWVG1bEk'
!mkdir -p deep_3drecon/BFM
!mv BFM/* deep_3drecon/BFM/ && rmdir BFM
!echo "BFM files:" && ls deep_3drecon/BFM/


## Cell 2b — Download Checkpoints

In [ ]:
# ── CELL 5: MimicTalk pretrained checkpoints ──
!gdown --folder '1Kc6ueDO9HFDN3BhtJCEKNCZtyKHSktaA'
!mkdir -p checkpoints checkpoints_mimictalk
!cp -r MimicTalk/checkpoints/*          checkpoints/
!cp -r MimicTalk/checkpoints_mimictalk/* checkpoints_mimictalk/
!rm -rf MimicTalk
print("checkpoints/")
!ls checkpoints/
print("checkpoints_mimictalk/")
!ls checkpoints_mimictalk/

## Cell 3 — Apply All Hotfixes

In [ ]:
# ── CELL 6: Apply all hotfixes ──
from pathlib import Path

# Fix 1: VisibleDeprecationWarning import crash
print("Fix 1: preprocess.py VisibleDeprecationWarning")
!sed -i "12s/.*VisibleDeprecationWarning.*/# &/" deep_3drecon/util/preprocess.py

# Fix 2+3: numpy.quantile import (image + video)
for path in [
    "data_gen/utils/process_image/fit_3dmm_landmark.py",
    "data_gen/utils/process_video/fit_3dmm_landmark.py",
]:
    print(f"Fix 2/3: {path}")
    !sed -i 's/^from numpy\.lib\.function_base import quantile$/from numpy import quantile/' "{path}" || true

# Fix 4: scatter_np shape unpack crash in mp_segmenter.py
print("Fix 4: scatter_np in mp_segmenter.py")
mp_path = Path("data_gen/utils/mp_feature_extractors/mp_segmenter.py")
assert mp_path.exists(), f"Not found: {mp_path}"

new_fn = r'''
def scatter_np(condition_img, classSeg=6):
    import numpy as np
    x = np.asarray(condition_img)
    if x.ndim == 2:
        x = x[None, None, :, :]
    elif x.ndim == 3:
        c = x.shape[-1]
        x = np.argmax(x, axis=-1) if c > 4 else x[..., 0]
        x = x[None, None, :, :]
    elif x.ndim == 4:
        if x.shape[1] != 1 and x.shape[-1] >= 1:
            c = x.shape[-1]
            x = np.argmax(x, axis=-1) if c > 4 else x[..., 0]
            x = x[:, None, :, :]
    elif x.ndim == 5:
        c = x.shape[-1]
        x = np.argmax(x, axis=-1) if c > 4 else x[..., 0]
    else:
        raise ValueError(f"scatter_np: unexpected shape {x.shape}")
    x = np.clip(x.astype(np.int64), 0, classSeg - 1)
    out = np.eye(classSeg, dtype=np.float32)[x[:, 0, :, :]]
    return out.transpose(0, 3, 1, 2)
'''

text  = mp_path.read_text()
lines = text.splitlines(True)
start = next(i for i, l in enumerate(lines) if l.startswith("def scatter_np"))
end   = next((j for j in range(start + 1, len(lines)) if lines[j].startswith("def ")), len(lines))
mp_path.write_text("".join(lines[:start]) + new_fn + "\n" + "".join(lines[end:]))

print("✅ All hotfixes applied.")


## Cell 3b — mmcv + setuptools + pkg_resources patches

In [ ]:
!pip install mmcv==2.2.0 \
    -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4.0/index.html \
    --no-build-isolation -q

import importlib, sys
importlib.invalidate_caches()
for k in [k for k in sys.modules if 'mmcv' in k]:
    del sys.modules[k]

try:
    import mmcv
    from mmcv.cnn import ConvModule
    print("mmcv", mmcv.__version__, "ready to train.")
except Exception as e:
    print("FAILED:", e)

# ── Fix: overwrite the broken system setuptools in-place ──
!pip install setuptools --upgrade \
    --target /usr/local/lib/python3.12/dist-packages/ \
    --ignore-installed -q

# Verify the broken line is gone
!python -c "import pkg_resources; print('pkg_resources OK')"
!python -c "import triton; print('triton OK')"
!python -c "import torchvision; print('torchvision OK')"

# ── Patch broken pkg_resources in-place (affects all subprocesses) ──
!sed -i \
  's/register_finder(pkgutil.ImpImporter, find_on_path)/pass  # patched: ImpImporter removed in py3.12/' \
  /usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py

# Confirm the patch is in the file
!grep "patched" /usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py

# Confirm subprocess sees it fixed now
!python -c "import torchvision; from mmcv.cnn import ConvModule; print('All OK — safe to train')"

## Cell 3c — pkg_resources / webrtcvad deep patches

In [ ]:
# ── Patch system pkg_resources for Python 3.12 compatibility ──
import subprocess

paths_to_patch = [
    "/usr/lib/python3/dist-packages/pkg_resources/__init__.py",
    "/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py",
]

for path in paths_to_patch:
    result = subprocess.run(f"grep -c 'ImpImporter' {path}", shell=True, capture_output=True, text=True)
    if result.stdout.strip() != "0":
        subprocess.run(
            f"sed -i 's/register_finder(pkgutil.ImpImporter, find_on_path)/pass  # patched: ImpImporter removed in Python 3.12/' {path}",
            shell=True
        )
        print(f"✅ Patched: {path}")
    else:
        print(f"⏭ Already clean: {path}")

# Verify
import importlib, sys
for mod in ["pkg_resources", "webrtcvad"]:
    if mod in sys.modules:
        del sys.modules[mod]
try:
    import webrtcvad
    print("✅ webrtcvad imports OK")
except Exception as e:
    print("❌", e)

# ── Force-reload pkg_resources after patch ──
import sys, pkgutil, importlib, types

# Fake ImpImporter on the live pkgutil module so cached code doesn't crash
if not hasattr(pkgutil, 'ImpImporter'):
    pkgutil.ImpImporter = type('ImpImporter', (), {})
    print("✅ Injected fake pkgutil.ImpImporter")

# Force reload both pkg_resources locations
for mod_name in list(sys.modules.keys()):
    if 'pkg_resources' in mod_name or 'webrtcvad' in mod_name:
        del sys.modules[mod_name]

import pkg_resources
import webrtcvad
print("✅ webrtcvad imports OK")

import subprocess

path = "/usr/lib/python3/dist-packages/pkg_resources/__init__.py"

# Check how many ImpImporter references remain
result = subprocess.run(f"grep -n 'ImpImporter' {path}", shell=True, capture_output=True, text=True)
print(result.stdout)

# Patch all remaining occurrences
subprocess.run(
    f"sed -i 's/register_namespace_handler(pkgutil.ImpImporter, file_ns_handler)/pass  # patched: ImpImporter removed in Python 3.12/' {path}",
    shell=True
)

# Verify none remain
result2 = subprocess.run(f"grep -n 'ImpImporter' {path}", shell=True, capture_output=True, text=True)
remaining = result2.stdout.strip()
if remaining:
    print("⚠️ Still remaining:\n", remaining)
else:
    print("✅ All ImpImporter references patched")

import subprocess

# 1. Patch webrtcvad.py — remove the pkg_resources import (it's only used for version checking)
ret = subprocess.run(
    "sed -i 's/import pkg_resources/# import pkg_resources  # patched for Python 3.12/' "
    "/usr/local/lib/python3.12/dist-packages/webrtcvad.py",
    shell=True, capture_output=True, text=True
)
print("webrtcvad patch:", "✅" if ret.returncode == 0 else "❌ " + ret.stderr)

# 2. Verify
result = subprocess.run(
    "grep -n 'pkg_resources' /usr/local/lib/python3.12/dist-packages/webrtcvad.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout or "✅ No pkg_resources references remain")

# 3. Quick import test in subprocess
test = subprocess.run("python -c \"import webrtcvad; print('webrtcvad OK')\"",
                      shell=True, capture_output=True, text=True)
print(test.stdout.strip() or test.stderr.strip())

import subprocess

subprocess.run(
    "sed -i \"s/__version__ = pkg_resources.get_distribution('webrtcvad').version/__version__ = '2.0.10'  # patched for Python 3.12/\" "
    "/usr/local/lib/python3.12/dist-packages/webrtcvad.py",
    shell=True
)

test = subprocess.run("python -c \"import webrtcvad; print('webrtcvad OK, version:', webrtcvad.__version__)\"",
                      shell=True, capture_output=True, text=True)
print(test.stdout.strip() or test.stderr.strip())

import subprocess

ret = subprocess.run(
    "pip install setuptools --upgrade "
    "--target /usr/lib/python3/dist-packages/ "
    "--ignore-installed -q",
    shell=True, capture_output=True, text=True
)
print("returncode:", ret.returncode)
print(ret.stderr[-300:] if ret.stderr else "no stderr")

# Verify
test = subprocess.run(
    "python -c \"import pkg_resources; import webrtcvad; import pygame; print('✅ all OK')\"",
    shell=True, capture_output=True, text=True
)
print(test.stdout.strip() or test.stderr[-500:])

import subprocess

path = "/usr/lib/python3/dist-packages/pkg_resources/__init__.py"

# Patch the find_module call on line ~2213
subprocess.run(
    f"sed -i 's/loader = importer.find_module(packageName)/loader = None  # patched: find_module removed in Python 3.12/' {path}",
    shell=True
)

# Clear all .pyc caches for pkg_resources
subprocess.run("find /usr/lib/python3/dist-packages/pkg_resources -name '*.pyc' -delete", shell=True)
subprocess.run("find /usr/lib/python3/dist-packages/__pycache__ -name 'pkg_resources*' -delete 2>/dev/null", shell=True)

# Verify
test = subprocess.run(
    "python -c \"import pkg_resources; import webrtcvad; import pygame; print('✅ all OK')\"",
    shell=True, capture_output=True, text=True
)
print(test.stdout.strip() or test.stderr[-500:])

## Cell 4 — Upload HDTF Prepared Data
> Upload `hdtf_prepared.zip` from Stage 1.

In [ ]:
import os, zipfile, glob
from google.colab import files

HDTF_BASE = '/content/hdtf_prepared'
os.makedirs(HDTF_BASE, exist_ok=True)

print('Upload hdtf_prepared.zip (from Stage 1)')
uploaded = files.upload()
for fname, data in uploaded.items():
    if fname.endswith('.zip'):
        zpath = f'/content/{fname}'
        with open(zpath, 'wb') as f: f.write(data)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(HDTF_BASE)
        print(f'✅ Extracted {fname}')

raw_vids = sorted(glob.glob(f'{HDTF_BASE}/raw/*.mp4'))
audios   = sorted(glob.glob(f'{HDTF_BASE}/audio/*.wav'))
print(f'\nRaw videos: {len(raw_vids)}')
print(f'Audio     : {len(audios)}')
for v in raw_vids:
    stem = os.path.splitext(os.path.basename(v))[0]
    has_audio = os.path.exists(f'{HDTF_BASE}/audio/{stem}.wav')
    print(f'  {stem}: video=✅ audio={"✅" if has_audio else "❌"}')

## Cell 5 — Batch Personalize + Generate (with resume)

In [ ]:
import os, time, glob, json, subprocess, shlex

HDTF_BASE   = '/content/hdtf_prepared'
MIMIC_OUT   = '/content/mimic_videos'
PROGRESS    = '/content/mimic_progress.json'
MAX_UPDATES = 2000

os.makedirs(MIMIC_OUT, exist_ok=True)

# ── Cross-identity audio mapping (cyclic: id01→id02, ..., id10→id01) ──
# This ensures lips are driven by DIFFERENT audio than the raw video's own audio,
# so LP-only baseline (which keeps the raw video's lip motion) will have BAD sync,
# while Hybrid-LP (MimicTalk-driven lips) will have GOOD sync.
def cross_audio_for(stem):
    num = int(stem.replace('id', ''))
    next_num = (num % 10) + 1
    return f'{HDTF_BASE}/audio/id{next_num:02d}.wav', f'id{next_num:02d}'

# Resume support
if os.path.exists(PROGRESS):
    with open(PROGRESS) as f: completed = json.load(f)
else:
    completed = {}

raw_vids = sorted(glob.glob(f'{HDTF_BASE}/raw/*.mp4'))
print(f'Found {len(raw_vids)} identities to process\n')
print('Cross-audio mapping:')
for vpath in raw_vids:
    stem = os.path.splitext(os.path.basename(vpath))[0]
    _, audio_id = cross_audio_for(stem)
    print(f'  {stem} video + {audio_id} audio')
print()

for vpath in raw_vids:
    stem = os.path.splitext(os.path.basename(vpath))[0]
    cross_audio_path, cross_audio_id = cross_audio_for(stem)
    final_out = f'{MIMIC_OUT}/{stem}.mp4'

    if stem in completed and os.path.exists(final_out):
        print(f'✓ {stem} (driven by {cross_audio_id}) already done')
        continue
    if not os.path.exists(cross_audio_path):
        print(f'⚠ {stem}: cross audio {cross_audio_id} not found, skipping')
        continue

    print(f'\n{"="*60}\n  {stem} ← driven by {cross_audio_id} audio\n{"="*60}')

    # ── 1) Personalize (uses identity's OWN video for visual learning) ──
    work_dir = f'checkpoints_mimictalk/{stem}'
    os.makedirs(work_dir, exist_ok=True)
    cfg_path = os.path.join(work_dir, 'config.yaml')

    if not os.path.exists(cfg_path):
        print(f'  [1/3] Personalizing on {stem} video ({MAX_UPDATES} updates)...')
        t0 = time.time()
        r = subprocess.run(
            f'python inference/train_mimictalk_on_a_video.py '
            f'--video_id {shlex.quote(vpath)} '
            f'--max_updates {MAX_UPDATES} '
            f'--work_dir {shlex.quote(work_dir)}',
            shell=True, capture_output=True, text=True)
        print(f'      ⏱ {(time.time()-t0)/60:.1f} min')
        if not os.path.exists(cfg_path):
            print(f'      ⚠ personalization failed: {r.stderr[-300:]}')
            continue
    else:
        print(f'  [1/3] Already personalized — reusing checkpoint')

    # ── 2) Prepare CROSS audio at 16kHz ──
    cross_16k = f'{HDTF_BASE}/audio/{cross_audio_id}_16k.wav'
    if not os.path.exists(cross_16k):
        subprocess.run(
            f'ffmpeg -y -i {shlex.quote(cross_audio_path)} -ac 1 -ar 16000 -sample_fmt s16 '
            f'{shlex.quote(cross_16k)}', shell=True, capture_output=True)

    # Style ref = silence (don't borrow style from cross audio source)
    style_16k = f'{HDTF_BASE}/raw/{stem}_style16k.wav'
    if not os.path.exists(style_16k):
        dur = subprocess.run(
            f'ffprobe -v error -show_entries format=duration -of csv=p=0 {shlex.quote(vpath)}',
            shell=True, capture_output=True, text=True).stdout.strip() or '30'
        subprocess.run(
            f'ffmpeg -y -f lavfi -i anullsrc=r=16000:cl=mono -t {dur} -sample_fmt s16 '
            f'{shlex.quote(style_16k)}', shell=True, capture_output=True)

    # ── 3) Inference: visual from {stem}, audio from CROSS identity ──
    print(f'  [2/3] Inference with CROSS audio ({cross_audio_id})...')
    os.makedirs('infer_out', exist_ok=True)
    raw_out = f'infer_out/{stem}_mimictalk.mp4'
    t0 = time.time()
    r = subprocess.run(
        f'python inference/mimictalk_infer.py '
        f'--torso_ckpt {shlex.quote(work_dir)} '
        f'--drv_aud {shlex.quote(cross_16k)} '
        f'--drv_pose {shlex.quote(vpath)} '
        f'--drv_style {shlex.quote(vpath)} '
        f'--out_name {shlex.quote(raw_out)} '
        f'--out_mode final',
        shell=True, capture_output=True, text=True)
    print(f'      ⏱ {time.time()-t0:.1f}s')

    if not os.path.exists(raw_out):
        cands = sorted(glob.glob('infer_out/**/*.mp4', recursive=True), key=os.path.getmtime)
        raw_out = cands[-1] if cands else None
    if not raw_out or not os.path.exists(raw_out):
        print(f'      ⚠ inference failed: {r.stderr[-300:]}')
        continue

    # ── 4) Upconvert to 30fps ──
    print(f'  [3/3] Upconverting to 30fps...')
    subprocess.run(
        f'ffmpeg -y -i {shlex.quote(raw_out)} '
        f'-filter:v "minterpolate=fps=30:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1" '
        f'-c:v libx264 -crf 18 -preset fast -c:a copy {shlex.quote(final_out)}',
        shell=True, capture_output=True)
    if not os.path.exists(final_out):
        import shutil; shutil.copy(raw_out, final_out)

    completed[stem] = {'video': vpath, 'cross_audio': cross_audio_path,
                       'cross_audio_id': cross_audio_id, 'output': final_out}
    with open(PROGRESS, 'w') as f: json.dump(completed, f, indent=2)
    print(f'  ✅ {stem} (driven by {cross_audio_id}) → {final_out}')

print(f'\n{"="*60}')
print(f'  ✅ Cross-audio batch complete: {len(completed)}/{len(raw_vids)} identities')
print(f'{"="*60}')


## Cell 6 — Package MimicTalk Videos

In [ ]:
import shutil, os, glob
from google.colab import files

MIMIC_OUT = '/content/mimic_videos'
vids = sorted(glob.glob(f'{MIMIC_OUT}/*.mp4'))
print(f'MimicTalk videos: {len(vids)}')
for v in vids:
    sz = os.path.getsize(v)//1024
    print(f'  {os.path.basename(v)} ({sz} KB)')

shutil.make_archive('/content/mimic_videos', 'zip', MIMIC_OUT)
sz = os.path.getsize('/content/mimic_videos.zip')//(1024*1024)
print(f'\n✅ Packaged: mimic_videos.zip ({sz} MB)')
files.download('/content/mimic_videos.zip')
print('\n✅ Stage 2 complete. Next: Stage 3 (LivePortrait batch)')